# LLM Intention Probing, Honesty, Deception, and Honest Mistakes, Algoverse 2026 Spring, KMSA & Tommy
## Part 1: Preparation

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # save plots to files only — do not display inline
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters
from utils.settings import *

# Utils
from utils.knowledge_check import (
    knowledge_check_truthfulqa, knowledge_check_mmlu,
    run_knowledge_check_truthfulqa, run_knowledge_check_mmlu,
)
from utils.generation import (
    generate_response,
    run_factual_generation, run_scenario_generation,
)
from utils.judge import (
    build_batch_requests_anthropic, parse_batch_results_anthropic,
    build_batch_requests_openai, parse_batch_results_openai,
    run_judge_anthropic, run_judge_openai,
    aggregate_judge_votes, build_full, print_threshold_summary,
)
from utils.activation import extract_activations, run_extract_activations, LABEL_MAP
from utils.analysis import (
    reduce_activations_pca, save_results_csv, select_pca_k, run_pca_reduction,
    filter_factual, build_probe_dataset,
)
from utils.probe import (
    probe_all_layers, probe_all_layers_binary,
    probe_all_layers_cascaded, probe_all_layers_mlp, probe_all_layers_cascaded_mlp,
)
from utils.plotting import (
    plot_macro_f1, plot_perclass_f1, plot_auroc, plot_top_confusion_matrices,
)

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Create output directories
for d in [
    KNOWLEDGE_TEST_DIR, RESPONSES_DIR,
    JUDGE_DIR, JUDGE_CLAUDE_HAIKU_DIR, JUDGE_GPT4O_MINI_DIR,
    OUTPUT_DIR, FIGURES_DIR,
    BINARY_DIR, TWAY_LR_DIR, TWAY_MLP_DIR, CASCADED_LR_DIR, CASCADED_MLP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Load fixed social scenario dataset
deception_df = pd.read_csv(DECEPTION_DATASET_PATH)
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts().to_string())

print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_ID}")

deception_dataset: (400, 6)
label
honest       200
deceptive    200

Device: cuda
GPU:  NVIDIA GeForce RTX 4090
VRAM: 25.3 GB
Model: Qwen/Qwen2.5-7B-Instruct


### 1.2 Load Model & Tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: "22GiB", "cpu": "120GiB"},
    offload_folder="outputs/offload",
)
model.eval()

# Support different config schemas (e.g., Gemma family and others).
cfg = getattr(model.config, "text_config", model.config)
N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
HIDDEN_DIM = getattr(cfg, "hidden_size", getattr(cfg, "d_model", None))

print(f"Loaded: {MODEL_ID}")
print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct
Layers: 28, hidden_dim: 3584


## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [3]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")

kc_tqa_df, tqa_passed_df, tqa_failed_df = run_knowledge_check_truthfulqa(
    tqa_mc, model, tokenizer, DEVICE, TRUTHFULQA_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(tqa_passed_df)} | Failed: {len(tqa_failed_df)}")

[skip] Already complete (817 rows): truthfulQA_test_results.csv

Passed: 338 | Failed: 479


### 2.2 MMLU

In [4]:
mmlu_mc = load_dataset("cais/mmlu", "all", split="test")

kc_mmlu_df, mmlu_passed_df, mmlu_failed_df = run_knowledge_check_mmlu(
    mmlu_mc, model, tokenizer, DEVICE, MMLU_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(mmlu_passed_df)} | Failed: {len(mmlu_failed_df)}")

[skip] Already complete (14042 rows): mmlu_test_results.csv

Passed: 4582 | Failed: 9460


## Part 3: Factual Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [5]:
tqa_resp_df = run_factual_generation(
    tqa_passed_df, tqa_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(tqa_resp_df["config"].value_counts().to_string())

[skip] Already complete (1155 rows): truthfulQA_responses.csv
config
B    479
A    338
C    338


#### 3.1.2 Claude Haiku Batch Judge

In [6]:
tqa_haiku_df = run_judge_anthropic(
    tqa_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_TQA_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_TQA_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

[skip] Already complete: judge_truthfulQA.csv


#### 3.1.3 GPT-4o-mini Batch Judge

In [7]:
tqa_gpt_df = run_judge_openai(
    tqa_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_TQA_PATH,
    state_path=JUDGE_GPT4O_MINI_TQA_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

[skip] Already complete: judge_truthfulQA.csv


### 3.2 MMLU Response Generation and Result Judge
#### 3.2.1 Response Generation

In [8]:
mmlu_resp_df = run_factual_generation(
    mmlu_passed_df, mmlu_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    MMLU_RESPONSES_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(mmlu_resp_df["config"].value_counts().to_string())

[skip] Already complete (18624 rows): mmlu_responses.csv
config
B    9460
A    4582
C    4582


#### 3.2.2 Claude Haiku Batch Judge

In [9]:
mmlu_haiku_df = run_judge_anthropic(
    mmlu_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_MMLU_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_MMLU_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

[skip] Already complete: judge_mmlu.csv


#### 3.2.3 GPT-4o-mini Batch Judge

In [10]:
mmlu_gpt_df = run_judge_openai(
    mmlu_resp_df,
    model=JUDGE_GPT4O_MINI_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_GPT4O_MINI_MMLU_PATH,
    state_path=JUDGE_GPT4O_MINI_MMLU_STATE,
    batch_dir=JUDGE_GPT4O_MINI_BATCH_DIR,
)

Resuming: 9 parts, 3 completed
[recover] Part 0: result file missing, re-downloading from OpenAI...
  → judge_mmlu_results_part0.jsonl
[recover] Part 1: result file missing, re-downloading from OpenAI...
  → judge_mmlu_results_part1.jsonl
[recover] Part 2: result file missing, re-downloading from OpenAI...
  → judge_mmlu_results_part2.jsonl
[recover] Rebuilding JSONL for pending parts: [4, 5, 6, 7, 8]
  → part 4: judge_mmlu_requests_part4.jsonl
  → part 5: judge_mmlu_requests_part5.jsonl
  → part 6: judge_mmlu_requests_part6.jsonl
  → part 7: judge_mmlu_requests_part7.jsonl
  → part 8: judge_mmlu_requests_part8.jsonl
[Part 3 | batch_69eaad7517408190bfa2dfe1fbd37a49] in_progress | completed=5926  total=5958
Waiting 180s...
[Part 3 | batch_69eaad7517408190bfa2dfe1fbd37a49] finalizing | completed=5958  total=5958
Waiting 180s...
[Part 3 | batch_69eaad7517408190bfa2dfe1fbd37a49] finalizing | completed=5958  total=5958
Waiting 180s...
[Part 3 | batch_69eaad7517408190bfa2dfe1fbd37a49] finali

### 3.3 Judge Result Compile

In [11]:
if TRUTHFULQA_FULL_PATH.exists() and MMLU_FULL_PATH.exists():
    tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
    mmlu_full = pd.read_csv(MMLU_FULL_PATH)
    print(f"Loaded tqa_full  ({len(tqa_full)} rows)")
    print(f"Loaded mmlu_full ({len(mmlu_full)} rows)")
else:
    tqa_votes  = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_TQA_PATH, JUDGE_GPT4O_MINI_TQA_PATH,
        vote_cols=VOTE_COLS,
    )
    mmlu_votes = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_MMLU_PATH, JUDGE_GPT4O_MINI_MMLU_PATH,
        vote_cols=VOTE_COLS,
    )
    tqa_full  = build_full(tqa_votes,  tqa_resp_df)
    mmlu_full = build_full(mmlu_votes, mmlu_resp_df)
    tqa_full.to_csv(TRUTHFULQA_FULL_PATH,  index=False)
    mmlu_full.to_csv(MMLU_FULL_PATH, index=False)
    print(f"Saved tqa_full  ({len(tqa_full)} rows) → {TRUTHFULQA_FULL_PATH.name}")
    print(f"Saved mmlu_full ({len(mmlu_full)} rows) → {MMLU_FULL_PATH.name}")

print_threshold_summary(tqa_full,  "TruthfulQA")
print_threshold_summary(mmlu_full, "MMLU")

Saved tqa_full  (1155 rows) → truthfulQA_full.csv
Saved mmlu_full (10946 rows) → mmlu_full.csv

=== TruthfulQA — votes_correct distribution ===
config           A    B    C
votes_correct               
0              191  309  287
1               41   48   22
2               32   32   11
3               33   43   12
4               15   23    2
5               12   12    1
6               14   12    3

  Config A (truth)       — useful range: votes_correct ≥ 4
    >= 6/6 correct :    14 rows
    >= 5/6 correct :    26 rows
    >= 4/6 correct :    41 rows

  Config B (honest_mistake) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   309 rows
    <= 1/6 correct :   357 rows
    <= 2/6 correct :   389 rows

  Config C (deception) — useful range: votes_correct ≤ 2
    <= 0/6 correct :   287 rows
    <= 1/6 correct :   309 rows
    <= 2/6 correct :   320 rows

=== MMLU — votes_correct distribution ===
config            A     B
votes_correct            
0              1638  2118
1    

## Part 4: Scenario Response Generation

In [12]:
scenario_resp_df = run_scenario_generation(
    deception_df, model, tokenizer, DEVICE,
    SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY,
    MAX_NEW_TOKENS, DO_SAMPLE,
)
print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
print(scenario_resp_df.head(2))

Starting fresh: 400 rows
400 rows to generate.


Scenario generation:   0%|          | 0/400 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generation complete.
Saved 200 pairs to scenario_responses.csv

Columns: ['pair_id', 'question', 'honest_scenario', 'honest_response', 'deceptive_scenario', 'deceptive_response']
   pair_id                                           question  \
0        0  Teacher: Why haven't you submitted your assign...   
1        1  Teacher: Why have you not submitted the scienc...   

                                     honest_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                     honest_response  \
0  Okay, the user is a student who's really stres...   
1  Okay, the user is a student who's procrastinat...   

                                  deceptive_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                  deceptive_response  
0  Okay, the user is a student who's really stres...  
1  Okay,

## Part 5: Build Probe Dataset and Extract Activations
### 5.1 Build Probe Dataset

In [13]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, scenario_resp_df, PROBE_DATASET_PATH,
)

Saved probe_dataset (3726 rows) → probe_dataset.csv

Label distribution:
label
honest_mistake    2427
truth              812
deception          487

Domain distribution:
domain
factual    3326
social      400


### 5.2 Extract Activations

In [14]:
activations_arr, labels_arr = run_extract_activations(
    probe_dataset, model, tokenizer, DEVICE,
    ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN, CHECKPOINT_EVERY,
)
print(f"\nLabel counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

Local files not found. Downloading from mikrokozmoz/algoverse_2026spring_kmsa_qwen3_4b_thinking_2507_activations ...
Download failed (RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69eb0166-7b0b4a0412451d323ff4d559;c433f429-2aa7-4f38-a870-7e05943c9d00)

Repository Not Found for url: https://huggingface.co/datasets/mikrokozmoz/algoverse_2026spring_kmsa_qwen3_4b_thinking_2507_activations/resolve/main/activations.npy.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication). Running extraction ...
Starting fresh: 3726 samples


Extracting activations:   0%|          | 0/3726 [00:00<?, ?it/s]

Extracted and saved: activations (3726, 36, 2560)

Label counts: {'truth': 812, 'honest_mistake': 2427, 'deception': 487}


## Part 6: Probe Training and Evaluation
### 6.1 Setup

In [15]:
labels_str = np.array([{v: k for k, v in LABEL_MAP.items()}[i] for i in labels_arr])

k_selection_df = select_pca_k(
    activations_arr, labels_str, PCA_K_VALUES, PCA_K_SELECTION_PATH,
)

n_layers=36, representative layers: 25%→layer 8, 50%→layer 17, 75%→layer 26
k values to scan: [16, 32, 64, 128, 256, 512]
Fitting PCA with max_k=512 per layer, then slicing for each k.

Layer 8 (25%):
  k=  16 | var=0.225 | val_F1=0.362 | train_F1=0.367 | gap=0.005 | time=0.1s
  k=  32 | var=0.316 | val_F1=0.378 | train_F1=0.394 | gap=0.016 | time=0.2s
  k=  64 | var=0.429 | val_F1=0.435 | train_F1=0.468 | gap=0.033 | time=0.4s
  k= 128 | var=0.560 | val_F1=0.488 | train_F1=0.556 | gap=0.068 | time=1.0s
  k= 256 | var=0.699 | val_F1=0.502 | train_F1=0.637 | gap=0.134 | time=8.6s
  k= 512 | var=0.833 | val_F1=0.486 | train_F1=0.775 | gap=0.289 | time=17.0s

Layer 17 (50%):
  k=  16 | var=0.258 | val_F1=0.458 | train_F1=0.467 | gap=0.009 | time=0.1s
  k=  32 | var=0.357 | val_F1=0.480 | train_F1=0.499 | gap=0.020 | time=0.2s
  k=  64 | var=0.477 | val_F1=0.532 | train_F1=0.556 | gap=0.024 | time=0.3s
  k= 128 | var=0.613 | val_F1=0.566 | train_F1=0.640 | gap=0.075 | time=1.5s
  k= 256 | 

In [16]:
acts_reduced = run_pca_reduction(
    activations_arr, PCA_K,
    ACTIVATIONS_PCA_PATH, PCA_COMPONENTS_PATH, PCA_VARIANCE_PATH,
    HF_ACTIVATIONS_REPO, HF_TOKEN,
)

Local files not found. Downloading from mikrokozmoz/algoverse_2026spring_kmsa_qwen3_4b_thinking_2507_activations ...
Download failed (404 Client Error. (Request ID: Root=1-69eb14a2-4a562fa81e137c79325f90b4;bd2f23df-8b41-425c-a173-6cf9eaeecf06)

Repository Not Found for url: https://huggingface.co/datasets/mikrokozmoz/algoverse_2026spring_kmsa_qwen3_4b_thinking_2507_activations/resolve/main/activations_pca64.npy.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication)
Running PCA (64 components) across 36 layers ...
Saved activations_pca64.npy       (3726, 36, 64)
Saved pca64_components.npy (36, 64, 2560)
Saved pca64_explained_variance.csv
Explained variance — mean: 0.462, min: 0.402, max: 0.575


### 6.2 Baseline: Binary Classifier

In [17]:
results_binary_c1 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=1.0,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C1_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C1.pkl",
)
results_binary_c01 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=0.1,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C01_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C01.pkl",
)

binary probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C1.csv (36 rows)


binary probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C01.csv (36 rows)


In [18]:
plot_auroc(
    [(results_binary_c1, "C=1.0"), (results_binary_c01, "C=0.1")],
    BINARY_DIR / "figures" / "auroc.png",
    title="Binary Probe AUROC per Layer (truth vs deception)",
)

Saved auroc.png


### 6.3 Approach 1: Direct 3-Way LR Classifier

In [19]:
results_3way_lr = probe_all_layers(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=TWAY_LR_PATH,
    checkpoint_path=TWAY_LR_DIR / "checkpoint_3way_lr.pkl",
)

3-way LR probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_3way_pca64.csv (36 rows)


In [20]:
plot_macro_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "macro_f1.png", title="3-Way LR: Macro F1 per Layer")
plot_perclass_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "perclass_f1.png", title="3-Way LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_lr, TWAY_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="LR ")

Saved macro_f1.png


Saved perclass_f1.png
Saved top5_cm.png


### 6.4 Approach 2: Direct 3-Way MLP Classifier

In [21]:
results_3way_mlp = probe_all_layers_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=TWAY_MLP_PATH,
    checkpoint_path=TWAY_MLP_DIR / "checkpoint_3way_mlp.pkl",
)

3-way MLP probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_3way_mlp_pca64.csv (36 rows)


In [22]:
plot_macro_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "macro_f1.png", title="3-Way MLP: Macro F1 per Layer")
plot_perclass_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "perclass_f1.png", title="3-Way MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_mlp, TWAY_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="MLP ")
plot_macro_f1(
    [(results_3way_lr, "LR"), (results_3way_mlp, "MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_lr_vs_mlp.png",
    title="3-Way Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_lr_vs_mlp.png


### 6.5 Approach 3: 2-Stage LR Classifier

In [23]:
results_cascaded_lr = probe_all_layers_cascaded(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=CASCADED_LR_PATH,
    checkpoint_path=CASCADED_LR_DIR / "checkpoint_cascaded_lr.pkl",
)

cascaded LR probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_cascaded_lr.csv (36 rows)


In [24]:
plot_macro_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "macro_f1.png", title="Cascaded LR: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "perclass_f1.png", title="Cascaded LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.6 Approach 4: 2-Stage MLP Classifier

In [25]:
results_cascaded_mlp = probe_all_layers_cascaded_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=CASCADED_MLP_PATH,
    checkpoint_path=CASCADED_MLP_DIR / "checkpoint_cascaded_mlp.pkl",
)

cascaded MLP probe (layers):   0%|          | 0/36 [00:00<?, ?it/s]

Saved probe_results_cascaded_mlp.csv (36 rows)


In [30]:
plot_macro_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "macro_f1.png", title="Cascaded MLP: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "perclass_f1.png", title="Cascaded MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded MLP ")
plot_macro_f1(
    [(results_cascaded_lr, "Cascaded LR"), (results_cascaded_mlp, "Cascaded MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_cascaded_lr_vs_mlp.png",
    title="Cascaded Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_cascaded_lr_vs_mlp.png


## Part 7: Model Comparison

In [31]:
# Load all probe results from CSV (safe to run after kernel restart)
r_lr    = pd.read_csv(TWAY_LR_PATH)
r_mlp   = pd.read_csv(TWAY_MLP_PATH)
r_clr   = pd.read_csv(CASCADED_LR_PATH)
r_cmlp  = pd.read_csv(CASCADED_MLP_PATH)
r_bin1  = pd.read_csv(BINARY_C1_PATH)
r_bin01 = pd.read_csv(BINARY_C01_PATH)

PROBE_RESULTS = [
    (r_lr,   "3-Way LR"),
    (r_mlp,  "3-Way MLP"),
    (r_clr,  "Cascaded LR"),
    (r_cmlp, "Cascaded MLP"),
]
SUMMARY_DIR = OUTPUT_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Loaded all probe results.")

Loaded all probe results.


In [32]:
layers = r_lr["layer"].values

# ── Table 1: Macro F1 per layer ───────────────────────────────────────────────
t1 = pd.DataFrame({"layer": layers})
for df, name in PROBE_RESULTS:
    t1[name] = df["f1_macro"].values
t1.to_csv(SUMMARY_DIR / "summary_macro_f1.csv", index=False)
print(t1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
for df, name in PROBE_RESULTS:
    ax.plot(layers, df["f1_macro"], marker="o", markersize=3, label=name)
ax.set_xlabel("Layer"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 per Layer — All Probes")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "macro_f1_all_probes.png", dpi=150); plt.close(fig)
print("Saved macro_f1_all_probes.png")

# ── Tables 2/3/4: Per-class F1 per layer ─────────────────────────────────────
for cls in ["truth", "honest_mistake", "deception"]:
    t = pd.DataFrame({"layer": layers})
    for df, name in PROBE_RESULTS:
        t[name] = df[f"f1_{cls}"].values
    t.to_csv(SUMMARY_DIR / f"summary_f1_{cls}.csv", index=False)
    print(f"\n── {cls} ──")
    print(t.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    for df, name in PROBE_RESULTS:
        ax.plot(layers, df[f"f1_{cls}"], marker="o", markersize=3, label=name)
    ax.set_xlabel("Layer"); ax.set_ylabel(f"F1 ({cls})")
    ax.set_title(f"{cls} F1 per Layer — All Probes")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(SUMMARY_DIR / f"f1_{cls}_all_probes.png", dpi=150); plt.close(fig)
    print(f"Saved f1_{cls}_all_probes.png")

# ── Table 5: AUROC per layer (binary baseline) ────────────────────────────────
t5 = pd.DataFrame({
    "layer":       r_bin1["layer"].values,
    "Binary C=1.0": r_bin1["auroc"].values,
    "Binary C=0.1": r_bin01["auroc"].values,
})
t5.to_csv(SUMMARY_DIR / "summary_auroc_binary.csv", index=False)
print("\n── Binary AUROC ──")
print(t5.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(r_bin1["layer"], r_bin1["auroc"], marker="o", markersize=3, label="C=1.0")
ax.plot(r_bin01["layer"], r_bin01["auroc"], marker="o", markersize=3, label="C=0.1")
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Binary Probe AUROC per Layer (truth vs deception)")
ax.set_ylim(0.5, 1.0); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "auroc_binary.png", dpi=150); plt.close(fig)
print("Saved auroc_binary.png")

 layer  3-Way LR  3-Way MLP  Cascaded LR  Cascaded MLP
     0  0.330976   0.395774     0.311740      0.378811
     1  0.349996   0.395442     0.330291      0.391250
     2  0.360156   0.398421     0.347497      0.412221
     3  0.379684   0.418445     0.380392      0.428836
     4  0.381681   0.431264     0.373982      0.436545
     5  0.385778   0.420259     0.389416      0.422509
     6  0.398229   0.417084     0.382324      0.423045
     7  0.404637   0.426099     0.392133      0.430715
     8  0.433369   0.440593     0.425680      0.437103
     9  0.446306   0.437624     0.437413      0.443313
    10  0.462472   0.446862     0.449907      0.450720
    11  0.465866   0.446883     0.461310      0.449120
    12  0.493259   0.449511     0.478119      0.463660
    13  0.485316   0.464411     0.486582      0.472196
    14  0.499457   0.469807     0.495451      0.475576
    15  0.509107   0.480781     0.490444      0.485312
    16  0.506094   0.480842     0.499810      0.478724
    17  0.

In [ ]:
# ── Upload large files to HuggingFace Hub ────────────────────────────────────
# Temporary: paste your token here; will be moved to settings.py later
HF_TOKEN = ""  # paste your token

from huggingface_hub import HfApi

api = HfApi()
files_to_upload = [
    ACTIVATIONS_PATH,
    LABELS_PATH,
    ACTIVATIONS_PCA_PATH,
    PCA_COMPONENTS_PATH,
]

for path in files_to_upload:
    print(f"Uploading {path.name} ...")
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=path.name,
        repo_id=HF_ACTIVATIONS_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    print(f"  done: {path.name}")

print("All uploads complete.")


Uploading activations.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  done: activations.npy
Uploading labels.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

  done: activations_pca64.npy
Uploading pca64_components.npy ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  done: pca64_components.npy
All uploads complete.
